# Notebook 4 – Date & Time Feature Engineering

Date and time feature engineering turns raw timestamps into numeric components that machine learning models can understand and use

**Dataset:** Online Retail Transactions (`data.csv`)
541,909 rows × 8 columns — InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country


In [23]:
import pandas as pd
df = pd.read_csv('data.csv', encoding='ISO-8859-1')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])   
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 1. Year, Month, Day

**Explanation:** Splits a date into its year, month, and day as separate numeric columns.

**Example:** `12/1/2010` → Year=2010, Month=12, Day=1

**Why we use / need:** Models can't read raw dates directly — breaking them into parts lets the model detect yearly, monthly, or daily patterns.

**When to use:** Use when you need to analyze trends by year, compare monthly performance, or study day-level patterns.

In [53]:
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Day'] = df['InvoiceDate'].dt.day
df[['InvoiceDate', 'Year', 'Month', 'Day']].head()

,InvoiceDate,Year,Month,Day
61619,2011-01-18 10:01:00,2011,1,18
61624,2011-01-18 10:17:00,2011,1,18
14938,2010-12-07 14:57:00,2010,12,7
14939,2010-12-07 14:57:00,2010,12,7
14940,2010-12-07 14:57:00,2010,12,7


**Code Explanation:** .dt.year, .dt.month, .dt.day extract each part from the datetime column.

## 2. Day of Week

**Explanation:** Identifies which day of the week a transaction happened.

**Example:** `12/1/2010` (Wednesday) → DayOfWeek = 2

**Why we use / need:** Customer buying habits often differ across weekdays vs weekends, so this helps the model learn weekly patterns.

**When to use:** Use in sales forecasting, footfall prediction, or any model where weekly cycles matter.

In [64]:
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek
df['DayName'] = df['InvoiceDate'].dt.day_name()
df[['InvoiceDate', 'DayOfWeek', 'DayName']].head()

,InvoiceDate,DayOfWeek,DayName
61619,2011-01-18 10:01:00,1,Tuesday
61624,2011-01-18 10:17:00,1,Tuesday
14938,2010-12-07 14:57:00,1,Tuesday
14939,2010-12-07 14:57:00,1,Tuesday
14940,2010-12-07 14:57:00,1,Tuesday


**Code Explanation:** `.dt.dayofweek` returns a number from 0 (Monday) to 6 (Sunday)

## 3. Week Number

**Explanation:** Shows which week of the year (1–53) a date falls into.

**Example:** 12/1/2010 → Week 48

**Why we use / need:** Weekly data smooths out daily noise while still capturing short-term trends, useful for tracking recurring events.

**When to use:** Use for weekly sales tracking, promotional campaign analysis, or weekly time-series forecasting.

In [39]:
df['WeekNumber'] = df['InvoiceDate'].dt.isocalendar().week
df[['InvoiceDate', 'WeekNumber']].head()

,InvoiceDate,WeekNumber
61619,2011-01-18 10:01:00,3
61624,2011-01-18 10:17:00,3
14938,2010-12-07 14:57:00,49
14939,2010-12-07 14:57:00,49
14940,2010-12-07 14:57:00,49


**Code Explanation:** `.dt.isocalendar().week` calculates the ISO week number.

## 4. Quarter

**Explanation:** Groups months into 4 business quarters (Q1–Q4).

**Example:** December → Quarter 4

**Why we use / need:** Businesses plan and report in quarters, so this feature aligns the model with real business cycles and reduces noise compared to raw month values.

**When to use:** Use for quarterly business reporting, financial forecasting, or when reducing granularity helps a simpler model.

In [54]:
df['Quarter'] = df['InvoiceDate'].dt.quarter
df[['InvoiceDate', 'Quarter']].head()

,InvoiceDate,Quarter
61619,2011-01-18 10:01:00,1
61624,2011-01-18 10:17:00,1
14938,2010-12-07 14:57:00,4
14939,2010-12-07 14:57:00,4
14940,2010-12-07 14:57:00,4


**Code Explanation:** `.dt.quarter` returns 1, 2, 3, or 4 based on the month.

## 5. Weekend Indicator

**Explanation:** A binary flag (1/0) showing if a purchase happened on a weekend.

**Example:** Saturday purchase → IsWeekend = 1

**Why we use / need:** Weekend behavior (leisure buying, bulk shopping) often differs sharply from weekdays, so a direct flag helps the model capture this effect easily.

**When to use:** Use when weekend vs weekday behavior is expected to influence the outcome, e.g. retail sales, restaurant footfall.

In [55]:
df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)
df[['InvoiceDate', 'DayOfWeek', 'IsWeekend']].head()

,InvoiceDate,DayOfWeek,IsWeekend
61619,2011-01-18 10:01:00,1,0
61624,2011-01-18 10:17:00,1,0
14938,2010-12-07 14:57:00,1,0
14939,2010-12-07 14:57:00,1,0
14940,2010-12-07 14:57:00,1,0


**Code Explanation:** Checks if DayOfWeek is 5 (Saturday) or 6 (Sunday), then converts True/False into 1/0.

## 6. Month Start / Month End

**Explanation:** Flags whether a date is the first or last day of its month

**Example:** `12/1/2010` → IsMonthStart = 1

**Why we use / need:** Sales often spike at month start (salary days) or month end (offers, deadlines), so this captures that business pattern

**When to use:** Use in retail, subscription, or salary-driven purchase behavior modeling

In [56]:
df['IsMonthStart'] = df['InvoiceDate'].dt.is_month_start.astype(int)
df['IsMonthEnd'] = df['InvoiceDate'].dt.is_month_end.astype(int)
df[['InvoiceDate', 'IsMonthStart', 'IsMonthEnd']].head()

,InvoiceDate,IsMonthStart,IsMonthEnd
61619,2011-01-18 10:01:00,0,0
61624,2011-01-18 10:17:00,0,0
14938,2010-12-07 14:57:00,0,0
14939,2010-12-07 14:57:00,0,0
14940,2010-12-07 14:57:00,0,0


**Code Explanation:** `.dt.is_month_start` / `.dt.is_month_end` return True/False, converted to 1/0

## 7. Days Since Event

**Explanation:** Measures how many days have passed since a fixed reference date.

**Example:** First transaction was `12/1/2010`; a purchase on `12/5/2010` → 4 days since event

**Why we use / need:** Converts a date into a continuous "time elapsed" number, which models understand better than a raw date.

**When to use:** Use for tracking time since signup, campaign launch, or any fixed starting event.

In [57]:
first_date = df['InvoiceDate'].min()
df['DaysSinceFirstEvent'] = (df['InvoiceDate'] - first_date).dt.days
df[['InvoiceDate', 'DaysSinceFirstEvent']].head()

,InvoiceDate,DaysSinceFirstEvent
61619,2011-01-18 10:01:00,48
61624,2011-01-18 10:17:00,48
14938,2010-12-07 14:57:00,6
14939,2010-12-07 14:57:00,6
14940,2010-12-07 14:57:00,6


**Code Explanation:** Subtracts the earliest date from each row's date, then `.dt.days` converts the result into a number.

## 8. Time Difference

**Explanation:** Calculates the gap in days between a customer's consecutive purchases.

**Example:** Purchases on Dec 1 and Dec 10 → TimeDiff = 9 days

**Why we use / need:** Helps detect buying frequency patterns per customer, useful in churn or repeat-purchase prediction.

**When to use:** Use in customer behavior analysis, churn modeling, or subscription renewal prediction.

In [69]:
df = df.sort_values(['CustomerID', 'InvoiceDate'])
df['TimeDiffDays'] = (
    df.groupby('CustomerID')['InvoiceDate']
      .diff()
      .dt.total_seconds() / (24 * 60 * 60)
)
df[['CustomerID', 'InvoiceDate', 'TimeDiffDays']].head(10)

,CustomerID,InvoiceDate,TimeDiffDays
61619,12346.0,2011-01-18 10:01:00,NaN
61624,12346.0,2011-01-18 10:17:00,0.011111
14938,12347.0,2010-12-07 14:57:00,NaN
14939,12347.0,2010-12-07 14:57:00,0.000000
14940,12347.0,2010-12-07 14:57:00,0.000000
14941,12347.0,2010-12-07 14:57:00,0.000000
14942,12347.0,2010-12-07 14:57:00,0.000000
14943,12347.0,2010-12-07 14:57:00,0.000000
14944,12347.0,2010-12-07 14:57:00,0.000000
14945,12347.0,2010-12-07 14:57:00,0.000000


**Code Explanation:** Sorts data by customer and date, then `.diff()` finds the gap between consecutive purchases per customer.

## 9. Customer Age

**Explanation:** Calculates age from date of birth (simulated here since the dataset lacks this column).

**Example:** DOB = 1985 → Age ≈ 41 (as of 2026)

**Why we use / need:** Age is a common demographic feature that affects buying behavior, product preference, and risk profiles.

**When to use:** Use in customer segmentation, credit scoring, insurance, or personalized recommendation models.

In [58]:
import numpy as np
np.random.seed(1)
dob = pd.Series(pd.to_datetime(np.random.choice(pd.date_range('1960-01-01', '2000-01-01'), size=len(df))))
df['CustomerAge'] = (pd.Timestamp('today') - dob).dt.days // 365
df[['CustomerAge']].head()

,CustomerAge
61619,64
61624,31
14938,27
14939,50
14940,62


**Code Explanation:** Generates random birth dates, subtracts from today's date, divides by 365 to get approximate age in years.

## 10. Customer Tenure

**Explanation:** Measures how many days a customer has been active since their first purchase.

**Example:** First purchase Dec 1, current row Dec 20 → Tenure = 19 days

**Why to use / need:** Long-tenured customers often behave differently (more loyal, higher lifetime value) than new customers, making this a key predictor.

**When we use:** Use in customer lifetime value (CLV) prediction, loyalty scoring, or churn modeling.

In [74]:
first_purchase = df.groupby('CustomerID')['InvoiceDate'].transform('min')
df['CustomerTenureDays'] = (
    df['InvoiceDate'] - first_purchase
).dt.days
df[['CustomerID', 'InvoiceDate', 'CustomerTenureDays']].head()

,CustomerID,InvoiceDate,CustomerTenureDays
61619,12346.0,2011-01-18 10:01:00,0.0
61624,12346.0,2011-01-18 10:17:00,0.0
14938,12347.0,2010-12-07 14:57:00,0.0
14939,12347.0,2010-12-07 14:57:00,0.0
14940,12347.0,2010-12-07 14:57:00,0.0


**Code Explanation:** `.transform('min')` assigns each row its customer's earliest purchase date without collapsing the data, then tenure is calculated by subtraction.

## 11. Recency

**Explanation:** Shows how many days ago a customer last purchased.

**Example:** Last purchase Nov 20, dataset's latest date Dec 9 → Recency = 19 days

**Why we use / need:** Recent buyers are more likely to be active/engaged; recency is a core part of the RFM (Recency, Frequency, Monetary) framework for customer segmentation.

**When to use:** Use in marketing analytics, churn prediction, and identifying active vs inactive customers.

In [60]:
snapshot_date = df['InvoiceDate'].max()
recency_df = df.groupby('CustomerID')['InvoiceDate'].max().reset_index()
recency_df['Recency'] = (snapshot_date - recency_df['InvoiceDate']).dt.days
recency_df.head()

,CustomerID,InvoiceDate,Recency
0,12346.0,2011-01-18 10:17:00,325
1,12347.0,2011-12-07 15:52:00,1
2,12348.0,2011-09-25 13:13:00,74
3,12349.0,2011-11-21 09:51:00,18
4,12350.0,2011-02-02 16:01:00,309


**Code Explanation:** Finds each customer's most recent purchase, then subtracts it from the overall latest date in the dataset.

## 12. Frequency

**Explanation:** Counts how many unique orders a customer placed.

**Example:** Customer with 5 invoice numbers → Frequency = 5

**Why we use / need:** Frequent buyers usually represent higher value and loyalty; frequency is another pillar of the RFM framework.

**When to use:** Use in customer segmentation, loyalty programs, and retention modeling.

In [48]:
frequency_df = df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
frequency_df.columns = ['CustomerID', 'Frequency']
frequency_df.head()

,CustomerID,Frequency
0,12346.0,2
1,12347.0,7
2,12348.0,4
3,12349.0,1
4,12350.0,1


**Code Explanation:** `.nunique()` counts distinct invoice numbers per customer, representing separate orders.

## 13. Seasonality

**Explanation:** Maps each month to a season (Winter, Spring, Summer, Autumn).

**Example:** December → Winter

**Why we use / need:** Many products have seasonal demand (e.g., coats in winter, ice cream in summer), so grouping months into seasons captures this domain knowledge for the model.

**When we use:** Use in demand forecasting, retail sales prediction, or any product with seasonal buying patterns.

In [49]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'
df['Season'] = df['Month'].apply(get_season)
df[['InvoiceDate', 'Month', 'Season']].head()

,InvoiceDate,Month,Season
61619,2011-01-18 10:01:00,1,Winter
61624,2011-01-18 10:17:00,1,Winter
14938,2010-12-07 14:57:00,12,Winter
14939,2010-12-07 14:57:00,12,Winter
14940,2010-12-07 14:57:00,12,Winter


**Code Explanation:** A custom function checks the month number and returns the matching season name, applied to every row using .apply()